<a href="https://colab.research.google.com/github/Nandita64/Auth/blob/main/AIML-Assignment%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk math collections sklearn pandas numpy

ERROR: Could not find a version that satisfies the requirement math (from versions: none)
ERROR: No matching distribution found for math


In [5]:
!pip install math

ERROR: Could not find a version that satisfies the requirement math (from versions: none)
ERROR: No matching distribution found for math


After installing, you might also need to download specific NLTK data resources, as indicated by the error you encountered earlier. For `punkt` and `stopwords`, you can use the following commands:

In [2]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from collections import defaultdict
import math
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Download: nltk.download('punkt'); nltk.download('stopwords')

def enhanced_summary(text, ratio=0.3):
    """
    Advanced TF-IDF
    """
    # Step 1: Tokenize sentences
    sentences = sent_tokenize(text)
    n_sentences = max(1, int(len(sentences) * ratio))


    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    processed_sents = []
    for sent in sentences:
        words = [stemmer.stem(word.lower()) for word in word_tokenize(sent)
                 if word.isalnum() and word.lower() not in stop_words]
        processed_sents.append(words)


    all_words = list(set(word for sent in processed_sents for word in sent))

    # Step 2-3: TF matrix (sent x words)
    tf_matrix = np.zeros((len(sentences), len(all_words)))
    for i, words in enumerate(processed_sents):
        for word in words:
            if word in all_words:
                j = all_words.index(word)
                tf_matrix[i, j] = 1 / len(words)
    tf_df = pd.DataFrame(tf_matrix, columns=all_words)

    # Step 4: IDF
    idf = np.zeros(len(all_words))
    N = len(sentences)
    for j, word in enumerate(all_words):
        doc_freq = sum(1 for words in processed_sents if word in words)
        idf[j] = math.log(N / (1 + doc_freq))
    idf_df = pd.DataFrame({'Word': all_words, 'IDF': idf})

    # Step 5: TF-IDF matrix
    tfidf_matrix = tf_matrix * idf
    tfidf_df = pd.DataFrame(tfidf_matrix, columns=all_words)

    # Sentence TF-IDF scores (row sums)
    sent_scores = np.sum(tfidf_matrix, axis=1)

    # Innovation: Cosine similarity to top TF-IDF words
    # Calculate the overall importance of each word (sum of TF-IDF scores across all sentences)
    word_scores_overall = np.sum(tfidf_matrix, axis=0)

    # Get indices of the top 5 words based on their overall importance
    top_word_indices = np.argsort(word_scores_overall)[-5:][::-1]

    # Create a single 'topic' vector that has the same number of features as sentence vectors
    # This vector represents the concept of the 'top words'.
    topic_vector_representation = np.zeros(len(all_words))
    # Set the values for the top words in this topic vector to 1 (or their weighted scores)
    topic_vector_representation[top_word_indices] = 1
    # Reshape it to be a 2D array (1, n_features) for cosine_similarity
    topic_vector_representation = topic_vector_representation.reshape(1, -1)

    # Calculate cosine similarity between each sentence's TF-IDF vector and this topic vector
    cos_sim = cosine_similarity(tfidf_matrix, topic_vector_representation).flatten()
    hybrid_scores = 0.7 * sent_scores + 0.3 * cos_sim  # Weighted hybrid

    # Step 6: Threshold (percentile for top n)
    threshold = np.percentile(hybrid_scores, 100 * (1 - ratio))

    # Step 7: Summary
    ranked_idx = np.argsort(hybrid_scores)[::-1]
    summary_idx = [i for i in ranked_idx if hybrid_scores[i] >= threshold][:n_sentences]
    summary_idx.sort()  # Original order
    summary = ' '.join([sentences[i] for i in summary_idx])

    return summary, threshold, {'TF': tf_df, 'IDF': idf_df, 'TFIDF': tfidf_df, 'Scores': dict(enumerate(hybrid_scores))}

# Sample cybersecurity text (longer for demo)
text = """
Cybersecurity threats have evolved dramatically with the rise of ransomware-as-a-service platforms on the dark web. Attackers now lease sophisticated encryption tools to amateurs, democratizing high-impact attacks against hospitals and schools. Blockchain technology offers promise for secure identity verification, yet smart contract vulnerabilities expose millions in cryptocurrency. Quantum computing looms as an existential risk, capable of shattering RSA encryption that protects global banking systems. Meanwhile, nation-state actors deploy supply chain attacks like SolarWinds, compromising thousands of enterprises through trusted software updates. Zero-trust architecture emerges as the defense paradigm, requiring continuous authentication across hybrid cloud environments. Machine learning anomaly detection systems flag 95% of intrusions within minutes, but suffer from high false positive rates in noisy environments. Behavioral analytics tracks user patterns to distinguish legitimate admins from compromised accounts. Privacy-preserving federated learning enables collaborative threat intelligence without exposing sensitive data. Container security addresses microservices risks through runtime monitoring and image scanning. Finally, human error remains the weakest link—phishing simulations reduce click rates by 70% after quarterly training. The cybersecurity arms race demands constant innovation against increasingly sophisticated adversaries.
"""

summary, thresh, matrices = enhanced_summary(text, ratio=0.3)
print("Summary:", summary)
print("Threshold:", round(thresh, 4))
print("\nTF Matrix (snippet):\n", matrices['TF'].head())
print("\nIDF:\n", matrices['IDF'])
print("\nSentence Scores:", {k: round(v, 3) for k, v in matrices['Scores'].items()})


Summary: 
Cybersecurity threats have evolved dramatically with the rise of ransomware-as-a-service platforms on the dark web. Container security addresses microservices risks through runtime monitoring and image scanning. The cybersecurity arms race demands constant innovation against increasingly sophisticated adversaries.
Threshold: 1.2388

TF Matrix (snippet):
    sensit  microservic  within   protect  tool  learn  feder  noisi   70  \
0     0.0          0.0     0.0  0.000000   0.0    0.0    0.0    0.0  0.0   
1     0.0          0.0     0.0  0.000000   0.1    0.0    0.0    0.0  0.0   
2     0.0          0.0     0.0  0.000000   0.0    0.0    0.0    0.0  0.0   
3     0.0          0.0     0.0  0.076923   0.0    0.0    0.0    0.0  0.0   
4     0.0          0.0     0.0  0.000000   0.0    0.0    0.0    0.0  0.0   

   architectur  ...  evolv       rsa  platform  compromis  admin  user  \
0          0.0  ...  0.125  0.000000     0.125   0.000000    0.0   0.0   
1          0.0  ...  0.000  